# Group Stats Visualization Template

Use `Group Info Scan.ipynb` for the configured report. This template uses the same Polars pipeline.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from wordcloud import STOPWORDS, WordCloud

from monthly_report.analysis import (
    build_message_dataframe,
    grouped_message_counts,
    monthly_platform_counts,
    top_n_with_other,
)
from monthly_report.config import ReportConfig
from monthly_report.loaders import load_all_messages
from monthly_report.text import load_userdict, split_words
from monthly_report.visualization import individual_colors, month_messages, recent_months

plt.style.use("dark_background")
# Initialized once per session. Rerunning a chart never changes a person's color.
INDIVIDUAL_COLORS = {}

In [ ]:
from matplotlib import font_manager as fm

font_path = r"<path to preferred font>"
font = fm.FontProperties(fname=font_path)
# title_font = fm.FontProperties(fname = font_path, size=42)
plt.rcParams["font.family"] = font.get_name()

In [ ]:
BASE_DIR = Path(r"<path to base directory>")
NAMEMAP_FILE = BASE_DIR / "<path to namemap>.json"
# Calendar-month cutoffs, recalculated whenever this cell runs.
TODAY = datetime.today()
MONTH_INDEX = TODAY.year * 12 + TODAY.month - 1
RECENT_CUTOFF = datetime((MONTH_INDEX - 3) // 12, (MONTH_INDEX - 3) % 12 + 1, 1)
OLD_CUTOFF = datetime((MONTH_INDEX - 9) // 12, (MONTH_INDEX - 9) % 12 + 1, 1)
TOP_N = 16

# Each page covers the next group of people, ranked by recent message count.
INDIVIDUAL_CLOUD_PAGES = 2
INDIVIDUAL_CLOUD_ROWS = 4
INDIVIDUAL_CLOUD_COLUMNS = 4
# None preserves all-history cloud text; use RECENT_CUTOFF for recent text only.
INDIVIDUAL_CLOUD_SINCE = None


ERASED_IDS = {
    "q(1234567890)",  # QQ
    "user76543210123",  # TG
}

config = ReportConfig(base_dir=BASE_DIR, namemap_file=NAMEMAP_FILE, erased_ids=ERASED_IDS)

messages = load_all_messages(config)
df = build_message_dataframe(messages)
if not INDIVIDUAL_COLORS:
    INDIVIDUAL_COLORS.update(individual_colors(df["name"].unique().to_list()))

In [ ]:
from monthly_report.loaders import load_namemap

nmp = load_namemap(config.namemap_file)
dfz = df.filter(~pl.col("id").is_in(list(config.erased_ids | nmp.keys())))
new_members = dfz.select("id", "name").unique(subset="id").sort("name", "id")

print(f"Members missing from namemap ({new_members.height}):")
for user_id, name in new_members.iter_rows():
    print(f"  {user_id}: {name}")
if new_members.is_empty():
    print("  None")


In [ ]:
counts_all = grouped_message_counts(df)
counts_recent = grouped_message_counts(df, since=RECENT_CUTOFF)
pie_all = top_n_with_other(counts_all, TOP_N)
pie_recent = top_n_with_other(counts_recent, TOP_N)

print(f"messages={len(messages):,}, members={df['name'].n_unique():,}")

In [ ]:
def draw_individual_pie(ax, counts, **kwargs):
    if counts.is_empty():
        ax.text(0.5, 0.5, "No messages", ha="center", va="center")
        ax.set_axis_off()
        return
    names = counts["name"].to_list()
    ax.pie(
        counts["msg_len"].to_numpy(),
        labels=names,
        colors=[INDIVIDUAL_COLORS[name] for name in names],
        autopct="%1.1f%%",
        **kwargs,
    )


fig, axes = plt.subplots(ncols=2, figsize=(16, 7), layout="constrained")
draw_individual_pie(axes[0], pie_all)
axes[0].set_title("Message Distribution: All")
draw_individual_pie(axes[1], pie_recent)
axes[1].set_title(f"Message Distribution: Since {RECENT_CUTOFF.date()}")
plt.show()

In [ ]:
month_periods, include_current_month = recent_months(datetime.today())
fig, axes = plt.subplots(3, 3, figsize=(15, 15), layout="constrained")

for ax, period in zip(axes.flat, month_periods, strict=True):
    counts = grouped_message_counts(month_messages(df, period))
    draw_individual_pie(
        ax,
        top_n_with_other(counts, TOP_N),
        textprops={"fontsize": 8},
        startangle=90,
    )
    ax.set_title(period.strftime("%Y-%m"))

window_start = month_periods[0].strftime("%Y-%m")
window_end = month_periods[-1].strftime("%Y-%m")
fig.suptitle(
    f"Sender Distribution by Month ({window_start} to {window_end})"
    f"\nInclude current month: {include_current_month}",
    fontsize=14,
)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")
monthly = monthly_platform_counts(df)
ax.bar(
    monthly["month"].to_list(),
    monthly.select(pl.sum_horizontal(pl.exclude("month"))).to_series().to_numpy(),
)
ax.tick_params(axis="x", rotation=90)
ax.set_title("# Message by Month")
ax.set_xlabel("Month")
plt.show()

In [ ]:
platform_monthly = monthly_platform_counts(df, since=OLD_CUTOFF)
fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
bottom = np.zeros(platform_monthly.height)
for platform in platform_monthly.columns:
    if platform == "month":
        continue
    values = platform_monthly[platform].to_numpy()
    ax.bar(platform_monthly["month"].to_list(), values, bottom=bottom, label=platform)
    bottom += values
ax.tick_params(axis="x", rotation=90)
ax.set_title("Platform Usage by Month")
ax.set_xlabel("Month")
if platform_monthly.width > 1:
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.show()

In [ ]:
plt.hist(df["date"].dt.hour().to_numpy(), bins=np.arange(0, 25))
plt.title("Hour Distribution")
plt.xlabel("Hour")
plt.show()

In [ ]:
# Word-cloud text preprocessing (ported from legacy notebook)
extra_sw = ["图片", "表情", "一个", "好像", "感觉", "说"]
STOPWORDS.update(extra_sw)

stopword_candidates = [
    Path("baidu_stopwords.txt"),
    BASE_DIR / "group info scan" / "baidu_stopwords.txt",
]
for sw_path in stopword_candidates:
    if sw_path.exists():
        with sw_path.open("r", encoding="utf-8") as f:
            STOPWORDS.update(line.strip() for line in f if line.strip())
        break

userdict_candidates = [
    Path("custom_dict.txt"),
    BASE_DIR / "group info scan" / "custom_dict.txt",
]
for dict_path in userdict_candidates:
    if dict_path.exists():
        load_userdict(dict_path)
        break


def make_wordcloud(messages, **kwargs):
    cloud = WordCloud(
        background_color="black",
        stopwords=STOPWORDS,
        collocations=False,
        font_path=font_path,
        relative_scaling=0.4,
        random_state=50,
        scale=2,
        **kwargs,
    )
    frequencies = cloud.process_text(split_words(messages))
    return cloud.generate_from_frequencies(frequencies) if frequencies else None

In [ ]:
wc = make_wordcloud(
    df.filter(pl.col("date") > RECENT_CUTOFF)["message"],
    width=1800,
    height=1800,
    max_words=200,
    max_font_size=300,
)
fig, ax = plt.subplots(figsize=(12, 8))
if wc is not None:
    ax.imshow(wc)
else:
    ax.text(0.5, 0.5, "No words", ha="center", va="center")
ax.set_title(f"Word Cloud: Messages Since {RECENT_CUTOFF.date()}")
ax.set_axis_off()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15), layout="constrained")
for ax, period in zip(axes.flat, month_periods, strict=True):
    month_wc = make_wordcloud(
        month_messages(df, period)["message"],
        width=1200,
        height=1200,
        max_words=133,
        max_font_size=270,
    )
    if month_wc is not None:
        ax.imshow(month_wc)
    else:
        ax.text(0.5, 0.5, "No words", ha="center", va="center")
    ax.set_axis_off()
    ax.set_title(period.strftime("%Y-%m"), fontsize=10)

fig.suptitle(
    f"Monthly Word Clouds ({window_start} to {window_end})"
    f"\nInclude current month: {include_current_month}",
    fontsize=14,
    y=1.02,
)
plt.show()

In [ ]:
if min(INDIVIDUAL_CLOUD_PAGES, INDIVIDUAL_CLOUD_ROWS, INDIVIDUAL_CLOUD_COLUMNS) < 1:
    raise ValueError("Cloud page count, rows and columns must be positive")

clouds_per_page = INDIVIDUAL_CLOUD_ROWS * INDIVIDUAL_CLOUD_COLUMNS
selected_chatters = counts_recent.head(INDIVIDUAL_CLOUD_PAGES * clouds_per_page)
cloud_frame = (
    df if INDIVIDUAL_CLOUD_SINCE is None else df.filter(pl.col("date") > INDIVIDUAL_CLOUD_SINCE)
)
# Group selected people's messages once rather than scanning the entire history per person.
chatter_texts = dict(
    cloud_frame.filter(pl.col("name").is_in(selected_chatters["name"].to_list()))
    .group_by("name")
    .agg(pl.col("message"))
    .iter_rows()
)

for start in range(0, selected_chatters.height, clouds_per_page):
    page = selected_chatters.slice(start, clouds_per_page)
    fig, axes = plt.subplots(
        INDIVIDUAL_CLOUD_ROWS,
        INDIVIDUAL_CLOUD_COLUMNS,
        figsize=(3.75 * INDIVIDUAL_CLOUD_COLUMNS, 3.75 * INDIVIDUAL_CLOUD_ROWS),
        layout="constrained",
        squeeze=False,
    )
    for ax, (name, recent_count) in zip(axes.flat, page.iter_rows(), strict=False):
        chatter_wc = make_wordcloud(
            pl.Series("message", chatter_texts.get(name, []), dtype=pl.String),
            width=900,
            height=900,
            max_words=133,
            max_font_size=200,
        )
        if chatter_wc is not None:
            ax.imshow(chatter_wc)
        else:
            ax.text(0.5, 0.5, "No words", ha="center", va="center")
        ax.set_title(f"{name} ({recent_count:,} recent messages)", fontsize=9)
        ax.set_axis_off()
    for ax in axes.flat[page.height :]:
        ax.set_axis_off()
    text_period = (
        "all history"
        if INDIVIDUAL_CLOUD_SINCE is None
        else f"since {INDIVIDUAL_CLOUD_SINCE.date()}"
    )
    fig.suptitle(
        f"Word Clouds: Chatters {start + 1}-{start + page.height} ({text_period})",
        fontsize=16,
        y=1.01,
    )
    plt.show()
if selected_chatters.is_empty():
    print("No chatters in the selected ranking period.")